# C02_P0_225 BigAlpha Transformer Submission

Single-file, self-contained inference notebook.
Paste the base64-encoded checkpoint into `MODEL_B64` before submission.


In [ ]:
import base64
import io
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

try:
    import dai
except Exception as exc:
    raise ImportError('dai is required in the BigQuant runtime') from exc

MODEL_B64 = """..."""
MODEL_PATH = Path('transformer_model.pt')
SEQ_LEN = 240
LOOKBACK_DAYS = 5
BUFFER_DAYS = 30
CHUNK_SIZE = 12
RUN_OFFICIAL_EVAL = True

PRICE_FIELDS = ('open', 'high', 'low', 'close')
VOLUME_FIELDS = ('volume', 'amount', 'deal_number')
COUNT_FIELDS = (
    'ask_num_orders1',
    'ask_num_orders2',
    'ask_num_orders3',
    'bid_num_orders1',
    'bid_num_orders2',
    'bid_num_orders3',
)


def _query(sql: str, *, filters: dict | None = None) -> pd.DataFrame:
    result = dai.query(sql, filters=filters or {}, compression=True)
    return result if isinstance(result, pd.DataFrame) else result.df()


def _normalize_universe(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['date'] = pd.to_datetime(out['date'], errors='coerce')
    out['instrument'] = out['instrument'].astype(str)
    return out.dropna(subset=['date', 'instrument']).drop_duplicates(['date', 'instrument'])


def _query_universe(table: str, start_date: str, end_date: str) -> pd.DataFrame:
    df = _query(
        f"""
        SELECT date, instrument
        FROM {table}
        WHERE DATE(date) >= DATE('{start_date}')
          AND DATE(date) <= DATE('{end_date}')
        ORDER BY date, instrument
        """
    )
    if df.empty:
        raise RuntimeError(f'empty universe returned for {table} {start_date}~{end_date}')
    return _normalize_universe(df)


def _universe_lookup(df: pd.DataFrame) -> set[tuple[pd.Timestamp, str]]:
    return {
        (pd.Timestamp(row.date).normalize(), str(row.instrument))
        for row in df.itertuples(index=False)
    }


def _available_columns(table: str, start_date: str, end_date: str) -> set[str]:
    sample = _query(
        f"""
        SELECT *
        FROM {table}
        WHERE DATE(date) >= DATE('{start_date}')
          AND DATE(date) <= DATE('{end_date}')
        LIMIT 1
        """
    )
    if sample is None or len(sample.columns) == 0:
        raise RuntimeError(f'empty or invalid schema returned for {table}')
    return set(sample.columns)


def _query_raw_bars(
    table: str,
    start_date: str,
    end_date: str,
    instruments: list[str] | None = None,
    feature_cols: tuple[str, ...] | None = None,
) -> pd.DataFrame:
    feature_cols = feature_cols or ()
    cols = _available_columns(table, start_date, end_date)
    missing = [field for field in feature_cols if field not in cols]
    if missing:
        raise ValueError(f'missing required feature columns in {table}: {missing}')
    select_fields = [field for field in feature_cols if field in cols]
    sql = f"SELECT date, instrument, {', '.join(select_fields)} FROM {table} ORDER BY instrument, date"
    filters = {'date': [start_date, end_date]}
    if instruments:
        filters['instrument'] = list(instruments)
    df = _query(sql, filters=filters)
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['instrument'] = df['instrument'].astype(str)
    for field in feature_cols:
        df[field] = pd.to_numeric(df[field], errors='coerce')
    return df.sort_values(['instrument', 'date']).reset_index(drop=True)


def _canonicalize(raw: pd.DataFrame, feature_cols: tuple[str, ...], preprocess_mode: str) -> pd.DataFrame:
    if raw.empty:
        return raw
    work = raw.copy()
    if preprocess_mode == 'fieldwise_log_zscore':
        for field in PRICE_FIELDS:
            valid = np.isfinite(work[field].to_numpy()) & (work[field].to_numpy() > 0)
            work[field] = np.where(valid, np.log(work[field]), np.nan)
        for field in VOLUME_FIELDS + COUNT_FIELDS:
            valid = np.isfinite(work[field].to_numpy()) & (work[field].to_numpy() >= 0)
            work[field] = np.where(valid, np.log1p(work[field]), np.nan)
    return work


def _session_close_mask(date_series: pd.Series) -> np.ndarray:
    session = date_series.dt.normalize().to_numpy()
    return np.concatenate([session[1:] != session[:-1], [True]])


def _build_windows(
    raw: pd.DataFrame,
    *,
    require_label: bool,
    feature_cols: tuple[str, ...],
    preprocess_mode: str = 'raw_zscore',
    universe_lookup: set[tuple[pd.Timestamp, str]] | None = None,
) -> tuple[pd.DataFrame, np.ndarray, np.ndarray | None]:
    if raw.empty:
        empty = pd.DataFrame(columns=['date', 'instrument', 'close'])
        return empty, np.empty((0, SEQ_LEN, len(feature_cols)), dtype=np.float32), None

    work = _canonicalize(raw, feature_cols, preprocess_mode)
    samples = []
    labels = []
    meta = []

    for instrument, group in work.groupby('instrument', sort=False):
        ordered = group.sort_values('date').reset_index(drop=True)
        if len(ordered) < SEQ_LEN:
            continue
        close_positions = np.flatnonzero(_session_close_mask(ordered['date']))
        if len(close_positions) < 2:
            continue
        feats = ordered.loc[:, feature_cols].to_numpy(np.float32)
        closes = ordered['close'].to_numpy(np.float32)
        sessions = ordered['date'].dt.normalize().to_numpy()
        for pos_idx, end_pos in enumerate(close_positions):
            if end_pos + 1 < SEQ_LEN:
                continue
            window_start = end_pos - SEQ_LEN + 1
            window_sessions = pd.unique(sessions[window_start : end_pos + 1])
            if len(window_sessions) < LOOKBACK_DAYS:
                continue
            window = feats[window_start : end_pos + 1]
            if window.shape[0] != SEQ_LEN or np.isnan(window).all():
                continue
            end_date = pd.Timestamp(ordered.loc[end_pos, 'date']).normalize()
            if universe_lookup is not None and (end_date, str(instrument)) not in universe_lookup:
                continue
            meta.append((end_date, instrument, float(closes[end_pos]) if np.isfinite(closes[end_pos]) else np.nan))
            samples.append(window)
            if require_label:
                if pos_idx + 1 >= len(close_positions):
                    labels.append(np.nan)
                else:
                    next_close_pos = close_positions[pos_idx + 1]
                    cur_close = closes[end_pos]
                    next_close = closes[next_close_pos]
                    if np.isfinite(cur_close) and np.isfinite(next_close) and cur_close != 0:
                        labels.append(next_close / cur_close - 1.0)
                    else:
                        labels.append(np.nan)

    if not samples:
        empty = pd.DataFrame(columns=['date', 'instrument', 'close'])
        return empty, np.empty((0, SEQ_LEN, len(feature_cols)), dtype=np.float32), None

    keys = pd.DataFrame(meta, columns=['date', 'instrument', 'close'])
    x = np.stack(samples).astype(np.float32)
    if require_label:
        y = np.asarray(labels, dtype=np.float32)
        valid = np.isfinite(y)
        return keys.loc[valid].reset_index(drop=True), x[valid], y[valid]
    return keys.reset_index(drop=True), x, None


def _filter_date_range(keys: pd.DataFrame, x: np.ndarray, y: np.ndarray | None, start_date: str, end_date: str):
    if keys.empty:
        return keys, x, y
    start_ts = pd.to_datetime(start_date).normalize()
    end_ts = pd.to_datetime(end_date).normalize()
    mask = (keys['date'] >= start_ts) & (keys['date'] <= end_ts)
    keys = keys.loc[mask].reset_index(drop=True)
    x = x[mask.to_numpy()]
    if y is None:
        return keys, x, y
    return keys, x, y[mask.to_numpy()]


def _standardize_windows(x: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    fill = mean.reshape(1, 1, -1)
    scale = std.reshape(1, 1, -1)
    x = np.where(np.isfinite(x), x, fill)
    return ((x - fill) / scale).astype(np.float32)


if torch is not None:
    class StockTransformer(nn.Module):
        def __init__(self, n_feat, d_model=96, nhead=4, nlayers=3, dim_ff=192, seq_len=SEQ_LEN):
            super().__init__()
            self._config = dict(n_feat=n_feat, d_model=d_model, nhead=nhead, nlayers=nlayers, dim_ff=dim_ff, seq_len=seq_len)
            self.proj = nn.Linear(n_feat, d_model)
            self.pos = nn.Parameter(torch.zeros(1, seq_len, d_model))
            layer = nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=dim_ff,
                dropout=0.1,
                batch_first=True,
                activation='gelu',
            )
            self.encoder = nn.TransformerEncoder(layer, nlayers)
            self.head = nn.Sequential(
                nn.LayerNorm(d_model),
                nn.Linear(d_model, d_model // 2),
                nn.GELU(),
                nn.Linear(d_model // 2, 1),
            )

        def forward(self, x):
            h = self.proj(x) + self.pos[:, : x.size(1)]
            h = self.encoder(h).mean(dim=1)
            return self.head(h).squeeze(-1)
else:
    class StockTransformer:  # type: ignore[no-redef]
        pass


def load_model(model_path: str | Path = MODEL_PATH, map_location='cpu') -> dict:
    if torch is None:
        raise ImportError('torch is required for inference')
    raw = base64.b64decode(MODEL_B64.encode('ascii'))
    buffer = io.BytesIO(raw)
    try:
        return torch.load(buffer, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(buffer, map_location=map_location)


def _predict_array(model: nn.Module, x: np.ndarray, device) -> np.ndarray:
    if len(x) == 0:
        return np.empty((0,), dtype=np.float32)
    out = []
    x_t = torch.from_numpy(x)
    model.eval()
    with torch.no_grad():
        for start in range(0, len(x_t), CHUNK_SIZE):
            xb = x_t[start : start + CHUNK_SIZE].to(device)
            out.append(model(xb).detach().cpu().numpy())
    return np.concatenate(out, axis=0).astype(np.float64)


In [ ]:
def validate_submission_output(df: pd.DataFrame, universe: pd.DataFrame, start_date: str, end_date: str) -> pd.DataFrame:
    required = ['date', 'instrument', 'score']
    if list(df.columns) != required:
        raise ValueError(f'submission output schema mismatch: {list(df.columns)}')
    out = df.copy()
    out['date'] = pd.to_datetime(out['date'], errors='coerce').dt.normalize()
    out['instrument'] = out['instrument'].astype(str)
    out['score'] = pd.to_numeric(out['score'], errors='coerce')
    if out['date'].isna().any():
        raise ValueError('submission output contains invalid dates')
    if out['instrument'].eq('').any():
        raise ValueError('submission output contains empty instrument codes')
    if not np.isfinite(out['score'].to_numpy()).all():
        raise ValueError('submission output contains non-finite scores')
    if out.duplicated(['date', 'instrument']).any():
        raise ValueError(f'submission output contains duplicate rows: {int(out.duplicated(["date", "instrument"]).sum())}')

    expected = _normalize_universe(universe[['date', 'instrument']].copy())
    expected = expected.loc[
        (expected['date'] >= pd.to_datetime(start_date).normalize())
        & (expected['date'] <= pd.to_datetime(end_date).normalize())
    ].reset_index(drop=True)
    if expected.empty:
        raise ValueError('universe keys are empty for validation')

    observed = out.drop_duplicates(['date', 'instrument']).reset_index(drop=True)
    expected_keys = pd.MultiIndex.from_frame(expected[['date', 'instrument']])
    observed_keys = pd.MultiIndex.from_frame(observed[['date', 'instrument']])
    missing_keys = expected_keys.difference(observed_keys)
    extra_keys = observed_keys.difference(expected_keys)
    if len(missing_keys) or len(extra_keys):
        raise ValueError(f'submission keys mismatch: missing={len(missing_keys)} extra={len(extra_keys)}')

    expected_dates = pd.Index(expected['date'].unique()).sort_values()
    observed_dates = pd.Index(observed['date'].unique()).sort_values()
    if not expected_dates.equals(observed_dates):
        raise ValueError('submission dates mismatch against universe')

    daily_std = out.groupby('date')['score'].std()
    if daily_std.isna().any() or (daily_std <= 0).any():
        bad_dates = daily_std.index[daily_std.isna() | (daily_std <= 0)].astype(str).tolist()
        raise ValueError(f'submission contains non-positive daily std for dates: {bad_dates[:10]}')
    total_std = float(out['score'].std())
    if not np.isfinite(total_std) or total_std <= 0:
        raise ValueError('submission score standard deviation must be positive')

    quantiles = out['score'].quantile([0.01, 0.05, 0.5, 0.95, 0.99]).to_dict()
    out.attrs['validation_report'] = {
        'score_quantiles': {str(k): float(v) for k, v in quantiles.items()},
        'total_std': total_std,
        'daily_std_min': float(daily_std.min()),
        'daily_std_mean': float(daily_std.mean()),
        'dates': int(out['date'].nunique()),
        'rows': int(len(out)),
    }
    return out.sort_values(['date', 'instrument']).reset_index(drop=True)


def predict_scores(model, table, start_date: str, end_date: str, instruments: list[str], stats, device, feature_cols: tuple[str, ...], preprocess_mode: str, universe_lookup: set[tuple[pd.Timestamp, str]]) -> pd.DataFrame:
    mean, std = stats
    mean = np.asarray(mean, dtype=np.float32)
    std = np.asarray(std, dtype=np.float32)
    rows = []
    query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=BUFFER_DAYS)).strftime('%Y-%m-%d %H:%M:%S')
    for i in range(0, len(instruments), CHUNK_SIZE):
        chunk = instruments[i : i + CHUNK_SIZE]
        raw = _query_raw_bars(table, query_start, end_date, chunk, feature_cols)
        keys, x, _ = _build_windows(raw, require_label=False, feature_cols=feature_cols, preprocess_mode=preprocess_mode, universe_lookup=universe_lookup)
        if len(keys) == 0:
            continue
        keys, x, _ = _filter_date_range(keys, x, None, start_date, end_date)
        if len(keys) == 0:
            continue
        x = _standardize_windows(x, mean, std)
        pred = _predict_array(model, x, device)
        out = keys[['date', 'instrument']].copy()
        out['score'] = pred
        rows.append(out)
    if not rows:
        return pd.DataFrame(columns=['date', 'instrument', 'score'])
    result = pd.concat(rows, ignore_index=True)
    result['date'] = pd.to_datetime(result['date'], errors='coerce').dt.normalize()
    result['instrument'] = result['instrument'].astype(str)
    result['score'] = pd.to_numeric(result['score'], errors='coerce').replace([np.inf, -np.inf], np.nan)
    result = result.dropna(subset=['date', 'instrument'])
    if result.duplicated(['date', 'instrument']).any():
        dup = result.loc[result.duplicated(['date', 'instrument'], keep=False), ['date', 'instrument']]
        raise ValueError(f'duplicate predictions detected: {len(dup)} rows')
    return result


def main(datasources, start_date: str, end_date: str) -> pd.DataFrame:
    if torch is None:
        raise ImportError('torch is required for inference')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ckpt = load_model(MODEL_PATH, map_location=device)
    feature_cols = tuple(ckpt.get('feature_cols') or ())
    if not feature_cols:
        raise ValueError('checkpoint is missing feature_cols')
    preprocess_mode = str(ckpt.get('preprocess_mode', 'raw_zscore'))
    model = StockTransformer(**ckpt['model_cfg']).to(device)
    model.load_state_dict(ckpt['state_dict'])
    model.eval()
    stats = (np.asarray(ckpt['mean'], dtype=np.float32), np.asarray(ckpt['std'], dtype=np.float32))

    table = datasources.get('bar5m') or datasources.get('bar_5m') or datasources.get('e2e_bar5m') or datasources.get('bar1m')
    if not table:
        raise ValueError('missing bar5m datasource')

    universe = _query_universe('bigalpha_2026_instruments', start_date, end_date)
    universe_lookup = _universe_lookup(universe)
    instruments = sorted(universe['instrument'].unique().tolist())
    score_df = predict_scores(model, table, start_date, end_date, instruments, stats, device, feature_cols, preprocess_mode, universe_lookup)

    result = universe.merge(score_df, on=['date', 'instrument'], how='left')
    missing_mask = result['score'].isna()
    coverage = (
        result.assign(missing_prediction=missing_mask)
        .groupby('date')
        .agg(
            universe_n=('instrument', 'size'),
            missing_n=('missing_prediction', 'sum'),
            missing_rate=('missing_prediction', 'mean'),
        )
        .reset_index()
    )
    if (coverage['universe_n'] - coverage['missing_n'] <= 0).any():
        bad_dates = coverage.loc[coverage['universe_n'] - coverage['missing_n'] <= 0, 'date'].astype(str).tolist()
        raise ValueError(f'one or more dates have zero valid predictions: {bad_dates[:10]}')
    valid_per_day = result.groupby('date')['score'].transform('count')
    if (valid_per_day == 0).any():
        raise ValueError('one or more dates have zero valid predictions')

    overall_missing_rate = float(missing_mask.mean()) if len(result) else 1.0
    mean_daily_missing_rate = float(coverage['missing_rate'].mean()) if len(coverage) else 1.0
    max_daily_missing_rate = float(coverage['missing_rate'].max()) if len(coverage) else 1.0
    if max_daily_missing_rate > 0.05:
        raise ValueError(f'daily missing prediction rate too high: {max_daily_missing_rate:.4f}')
    daily_mean = result.groupby('date')['score'].transform('mean')
    filled_mask = result['score'].isna()
    if filled_mask.any() and daily_mean[filled_mask].isna().any():
        raise ValueError('one or more dates have no valid score mean for filling')
    result['score'] = result['score'].fillna(daily_mean)
    prediction_report = {
        'overall_missing_rate': overall_missing_rate,
        'mean_daily_missing_rate': mean_daily_missing_rate,
        'max_daily_missing_rate': max_daily_missing_rate,
        'dates_above_1pct': coverage.loc[coverage['missing_rate'] > 0.01, 'date'].astype(str).tolist(),
        'dates_above_5pct': coverage.loc[coverage['missing_rate'] > 0.05, 'date'].astype(str).tolist(),
        'filled_score_count': int(filled_mask.sum()),
        'filled_score_rate': float(filled_mask.mean()) if len(result) else 0.0,
        'filled_dates': sorted(result.loc[filled_mask, 'date'].astype(str).unique().tolist()),
        'coverage_status': 'PASS' if max_daily_missing_rate < 0.01 else ('WARNING' if max_daily_missing_rate <= 0.05 else 'FAIL'),
    }
    result = validate_submission_output(result, universe, start_date, end_date)
    result.attrs['prediction_report'] = prediction_report
    return result


def _static_contract_check():
    # Runtime-safe contract check: do not read any local notebook or source file.
    required_objects = {
        'main': main,
        'StockTransformer': StockTransformer,
        'load_model': load_model,
        'predict_scores': predict_scores,
        'validate_submission_output': validate_submission_output,
    }
    missing = [name for name, obj in required_objects.items() if obj is None]
    if missing:
        raise ValueError(f'missing required runtime objects: {missing}')
    if not callable(main):
        raise TypeError('main must be callable')
    if not isinstance(RUN_OFFICIAL_EVAL, bool):
        raise TypeError('RUN_OFFICIAL_EVAL must be bool')


In [ ]:
if not MODEL_B64.strip() or 'PASTE_THE_BASE64' in MODEL_B64:
    raise ValueError('MODEL_B64 is empty; paste the base64-encoded transformer_model.pt bytes into the notebook')

_static_contract_check()

datasources = {'bar5m': 'bigalpha_2026_stock_bar5m', 'bar1m': 'bigalpha_2026_stock_bar1m'}
start_date = '2024-01-01 00:00:00'
end_date = '2024-12-31 23:59:59'
score_data = main(datasources, start_date, end_date)
print(score_data.head())
print({'prediction_report': score_data.attrs.get('prediction_report', {})})

if RUN_OFFICIAL_EVAL:
    from bigmodule import M
    result = M.bigalpha_eval._latest(factor_data=score_data, show=True)
else:
    print('LOCAL DRY RUN ONLY')
